In [ ]:
import marimo as mo
import polars as pl
import gzip
import re
import xml.etree.ElementTree as ET
from collections import defaultdict, deque
from pathlib import Path
from urllib.parse import quote_plus

In [ ]:
CATALOG_CSV = Path("../downloaded/mmrrc_catalog_data.csv.gz")

catalog = pl.read_csv(CATALOG_CSV).rename(str.strip)
mo.md(f"**{CATALOG_CSV.name}**: {catalog.height:,} rows x {catalog.width} columns")

**mmrrc_catalog_data.csv.gz**: 588,980 rows x 18 columns

In [ ]:
catalog_table = mo.ui.table(catalog, page_size=20)
catalog_table

_marimo_row_id,STRAIN/STOCK_ID,STRAIN/STOCK_DESIGNATION,OTHER_NAMES,STRAIN_TYPE,STATE,MGI_ALLELE_ACCESSION_ID,ALLELE_SYMBOL,ALLELE_NAME,MUTATION_TYPE,CHROMOSOME,MGI_GENE_ACCESSION_ID,GENE_SYMBOL,GENE_NAME,SDS_URL,ACCEPTED_DATE,MPT_IDS,PUBMED_IDS,RESEARCH_AREAS
u32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
0,"""MMRRC:000001-UNC""","""C57BL/6-Tg(Fga,Fgb,Fgg)1Unc/Mm…","""RRID:MMRRC_000001-UNC""","""MSR""","""CA""",null,null,null,"""TG""","""3""","""MGI:95526""","""Fgg""","""fibrinogen gamma chain""","""https://www.mmrrc.org/catalog/…","""05/01/2001""",null,"""PMID: 11521996""",null
1,"""MMRRC:000001-UNC""","""C57BL/6-Tg(Fga,Fgb,Fgg)1Unc/Mm…","""RRID:MMRRC_000001-UNC""","""MSR""","""CA""",null,null,null,"""TG""","""3""","""MGI:99501""","""Fgb""","""fibrinogen beta chain""","""https://www.mmrrc.org/catalog/…","""05/01/2001""",null,"""PMID: 11521996""",null
2,"""MMRRC:000001-UNC""","""C57BL/6-Tg(Fga,Fgb,Fgg)1Unc/Mm…","""RRID:MMRRC_000001-UNC""","""MSR""","""CA""",null,null,null,"""TG""","""3""","""MGI:1316726""","""Fga""","""fibrinogen alpha chain""","""https://www.mmrrc.org/catalog/…","""05/01/2001""",null,"""PMID: 11521996""",null
3,"""MMRRC:000001-UNC""","""C57BL/6-Tg(Fga,Fgb,Fgg)1Unc/Mm…","""RRID:MMRRC_000001-UNC""","""MSR""","""CA""","""MGI:3696864""","""Tg(Fga,Fgb,Fgg)1Unc""","""transgene insertion 1, Univers…","""TG""","""unknown""",null,null,null,"""https://www.mmrrc.org/catalog/…","""05/01/2001""",null,"""PMID: 11521996""",null
4,"""MMRRC:000002-UNC""","""B6.129P2-<i>Esr2<sup>tm1Unc</s…","""RRID:MMRRC_000002-UNC""","""CON""","""CA""","""MGI:2152217""","""Esr2<tm1Unc>""","""estrogen receptor 2 (beta); ta…","""TM""","""12""",null,null,null,"""https://www.mmrrc.org/catalog/…","""04/24/2001""","""decreased bone mineral density…","""PMID: 9861029""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
588975,"""MMRRC:076502-UCD""","""C57BL/6NCrl-<i>4833439L19Rik<s…","""RRID:MMRRC_076502-UCD, CR11796""","""COI""","""CU,SP""",null,null,null,null,"""13""","""MGI:1921162""","""4833439L19Rik""","""RIKEN cDNA 4833439L19 gene""","""https://www.mmrrc.org/catalog/…","""07/10/2026""",null,null,null
588976,"""MMRRC:076503-UCD""","""C57BL/6NCrl-<i>Pdrg1<sup>em1(I…","""RRID:MMRRC_076503-UCD, CR11781""","""COI""","""CU,SP""",null,"""Pdrg1<em1(IMPC)Mbp>""","""p53 and DNA damage regulated 1…",null,"""2""",null,null,null,"""https://www.mmrrc.org/catalog/…","""07/13/2026""",null,null,null
588977,"""MMRRC:076503-UCD""","""C57BL/6NCrl-<i>Pdrg1<sup>em1(I…","""RRID:MMRRC_076503-UCD, CR11781""","""COI""","""CU,SP""",null,null,null,null,"""2""","""MGI:1915809""","""Pdrg1""","""p53 and DNA damage regulated 1""","""https://www.mmrrc.org/catalog/…","""07/13/2026""",null,null,null


In [ ]:
_ci_strains = catalog.filter(pl.col("MUTATION_TYPE") == "CI")["STRAIN/STOCK_ID"].n_unique()
_ci_rows = catalog.filter(pl.col("MUTATION_TYPE") == "CI").height
_ci_genes = (
    catalog.filter(pl.col("MUTATION_TYPE") == "CI")
    .group_by("STRAIN/STOCK_ID")
    .agg(pl.col("GENE_SYMBOL").drop_nulls().n_unique().alias("g"))["g"]
    .mean()
)

# MMRRC mutation-type legend, from https://www.mmrrc.org/methods/data_download.php
MUTATION_LABELS = {
    "Targeted mutation (TM)": "TM",
    "Gene trap (GT)": "GT",
    "Transgenic (TG)": "TG",
    "Deletion (DEL)": "DEL",
    "Spontaneous (SM)": "SM",
    "Insertion (INS)": "INS",
    "Inversion (INV)": "INV",
    "Duplication (DP)": "DP",
    "Transposition (TP)": "TP",
    "Chromosomal aberration (CH)": "CH",
    "Radiation induced (RAD)": "RAD",
    "Chromosomal segment (CS)": "CS",
    "Other (OTH)": "OTH",
    "No mutation type recorded": "(none)",
    "Chemically induced / ENU (CI)": "CI",
}

mutation_filter = mo.ui.multiselect(
    options=MUTATION_LABELS,
    value=[_k for _k in MUTATION_LABELS if MUTATION_LABELS[_k] != "CI"],
    label="Mutation types to include",
)

mo.vstack([
    mo.md(f"""
# Gene exploration

Navigate the catalog gene-first: which genes are manipulated across the most
strains, how that breaks down by chromosome, and where to follow each gene up
at MGI and NCBI.

## Why chemically-induced strains are excluded by default

**{_ci_strains:,} chemically-induced (ENU) strains carry a mean of
{_ci_genes:.1f} genes each**, and between them they produce {_ci_rows:,} of the
catalog's {catalog.height:,} rows ({_ci_rows / catalog.height:.0%}). Every other
mutation type averages one to three genes per strain.

That is because an ENU strain's gene list is a set of *candidate variants found
by sequencing*, not a set of deliberate manipulations. Longer genes accumulate
more random ENU hits simply by being bigger targets, so ranking genes by strain
count with `CI` included returns Ttn, Obscn, Neb, Macf1, Hmcn1 and Syne2 — the
longest genes in the mouse genome. That ranking measures gene length, not
research interest.

So `CI` starts unticked and the tables below describe deliberately manipulated
genes. **Tick it back on to see the ENU picture** — the ranking changes
completely, which is itself the point.
"""),
    mutation_filter,
])

Gene exploration 
 Navigate the catalog gene-first: which genes are manipulated across the most
strains, how that breaks down by chromosome, and where to follow each gene up
at MGI and NCBI. 
 Why chemically-induced strains are excluded by default 
 8,229 chemically-induced (ENU) strains carry a mean of
58.1 genes each , and between them they produce 478,364 of the
catalog's 588,980 rows (81%). Every other
mutation type averages one to three genes per strain. 
 That is because an ENU strain's gene list is a set of candidate variants found
by sequencing , not a set of deliberate manipulations. Longer genes accumulate
more random ENU hits simply by being bigger targets, so ranking genes by strain
count with CI included returns Ttn, Obscn, Neb, Macf1, Hmcn1 and Syne2 — the
longest genes in the mouse genome. That ranking measures gene length, not
research interest. 
 So CI starts unticked and the tables below describe deliberately manipulated
genes. Tick it back on to see the ENU picture — the ranking changes
completely, which is itself the point. <marimo-multiselect data-initial-value='["Targeted mutation (TM)","Gene trap (GT)","Transgenic (TG)","Deletion (DEL)","Spontaneous (SM)","Insertion (INS)","Inversion (INV)","Duplication (DP)","Transposition (TP)","Chromosomal aberration (CH)","Radiation induced (RAD)","Chromosomal segment (CS)","Other (OTH)","No mutation type recorded"]' data-label='"\u003cspan class=\"markdown prose dark:prose-invert contents\"\u003e\u003cspan class=\"paragraph\"\u003eMutation types to include\u003c/span\u003e\u003c/span\u003e"' data-options='["Targeted mutation (TM)","Gene trap (GT)","Transgenic (TG)","Deletion (DEL)","Spontaneous (SM)","Insertion (INS)","Inversion (INV)","Duplication (DP)","Transposition (TP)","Chromosomal aberration (CH)","Radiation induced (RAD)","Chromosomal segment (CS)","Other (OTH)","No mutation type recorded","Chemically induced / ENU (CI)"]' data-full-width='false' data-disabled='false'>

In [ ]:
MOUSE_CHROMS = [str(_i) for _i in range(1, 20)] + ["X", "Y", "MT"]

# CHROMOSOME is free text: 50 distinct values for what should be 22, including
# "Chr 1", "Chr11:4938754-4948064 bp", "8q21.13", "unk" and "N/A". Recover what
# is recoverable and bucket the rest as "unmapped".
_chrom = (
    pl.col("CHROMOSOME")
    .str.strip_chars()
    .str.to_uppercase()
    .str.replace(r"^CHR\s*", "")
    .str.strip_chars()
)

# Rows come in two disjoint kinds: gene rows (GENE_SYMBOL and
# MGI_GENE_ACCESSION_ID set, allele columns null) and allele rows (the reverse).
# Zero rows carry both, so a gene is linked to its alleles only via the strain.
catalog_genes = catalog.filter(pl.col("GENE_SYMBOL").is_not_null()).with_columns(
    pl.when(_chrom.is_in(MOUSE_CHROMS))
    .then(_chrom)
    .otherwise(pl.lit("unmapped"))
    .alias("chrom"),
    pl.col("MUTATION_TYPE").fill_null("(none)").alias("mut"),
)

strain_alleles = (
    catalog.filter(pl.col("ALLELE_SYMBOL").is_not_null())
    .select(["STRAIN/STOCK_ID", "ALLELE_SYMBOL", "MGI_ALLELE_ACCESSION_ID"])
    .unique()
)

mo.md(
    f"`catalog_genes`: {catalog_genes.height:,} gene rows "
    f"({catalog_genes['GENE_SYMBOL'].n_unique():,} distinct symbols) &nbsp;·&nbsp; "
    f"`strain_alleles`: {strain_alleles.height:,} allele rows "
    f"({strain_alleles['ALLELE_SYMBOL'].n_unique():,} distinct alleles)"
)

`catalog_genes`: 534,057 gene rows (24,593 distinct symbols) &nbsp;·&nbsp; `strain_alleles`: 44,586 allele rows (28,406 distinct alleles)

In [ ]:
def mgi_url(mgi_id):
    """Link to an MGI marker detail page, e.g. MGI:98864 -> .../marker/MGI:98864."""
    return f"https://www.informatics.jax.org/marker/{mgi_id}"


def ncbi_url(symbol, mgi_id=None):
    """Link to an NCBI Gene record via search.

    The catalog carries no Entrez id, but `Ttn[sym] AND "Mus musculus"[orgn]`
    resolves straight to the record page rather than a result list. Symbols with
    no MGI id are mostly non-mouse transgenes (SOD1, APP), which would not match
    the mouse organism clause, so it is dropped for them.
    """
    _term = f"{symbol}[sym]"
    if mgi_id is not None:
        _term += ' AND "Mus musculus"[orgn]'
    return "https://www.ncbi.nlm.nih.gov/gene/?term=" + quote_plus(_term)


_selected_muts = mutation_filter.value
_kept = catalog_genes.filter(pl.col("mut").is_in(_selected_muts))

# Alleles reach a gene only through the strains that carry it (see catalog_genes).
_gene_alleles = (
    _kept.select(["GENE_SYMBOL", "STRAIN/STOCK_ID"])
    .unique()
    .join(strain_alleles, on="STRAIN/STOCK_ID", how="left")
    .group_by("GENE_SYMBOL")
    .agg(pl.col("ALLELE_SYMBOL").drop_nulls().n_unique().alias("alleles"))
)

# Kept unfiltered so the gap against `strains` shows how much of a gene's
# presence is ENU noise.
_all_strains = catalog_genes.group_by("GENE_SYMBOL").agg(
    pl.col("STRAIN/STOCK_ID").n_unique().alias("strains_all")
)

gene_index = (
    _kept.group_by("GENE_SYMBOL")
    .agg(
        pl.col("STRAIN/STOCK_ID").n_unique().alias("strains"),
        pl.col("chrom").first().alias("chrom"),
        pl.col("GENE_NAME").drop_nulls().first().alias("gene_name"),
        pl.col("MGI_GENE_ACCESSION_ID").drop_nulls().first().alias("mgi_id"),
        pl.col("mut").unique().sort().str.join(", ").alias("mutation_types"),
    )
    .join(_gene_alleles, on="GENE_SYMBOL", how="left")
    .join(_all_strains, on="GENE_SYMBOL", how="left")
    .rename({"GENE_SYMBOL": "gene_symbol"})
    .with_columns(
        pl.struct(["gene_symbol", "mgi_id"])
        .map_elements(
            lambda r: ncbi_url(r["gene_symbol"], r["mgi_id"]), return_dtype=pl.String
        )
        .alias("ncbi")
    )
    .select([
        "gene_symbol", "gene_name", "chrom", "strains", "strains_all",
        "alleles", "mutation_types", "mgi_id", "ncbi",
    ])
    .sort("strains", descending=True)
)

mo.md(
    f"`gene_index`: **{gene_index.height:,} genes** under the current mutation-type "
    f"filter, out of {catalog_genes['GENE_SYMBOL'].n_unique():,} in the catalog."
)

`gene_index`: **16,426 genes** under the current mutation-type filter, out of 24,593 in the catalog.

In [ ]:
_ck = catalog_genes.filter(pl.col("mut").is_in(mutation_filter.value))

# Sort naturally (1..19, X, Y, MT, unmapped) rather than by count, so the table
# reads like a karyotype; it is still click-sortable by any column.
_order = {_c: _i for _i, _c in enumerate(MOUSE_CHROMS + ["unmapped"])}

chrom_summary = (
    _ck.group_by("chrom")
    .agg(
        pl.col("GENE_SYMBOL").n_unique().alias("genes"),
        pl.col("STRAIN/STOCK_ID").n_unique().alias("strains"),
    )
    .join(
        _ck.group_by(["chrom", "GENE_SYMBOL"])
        .agg(pl.col("STRAIN/STOCK_ID").n_unique().alias("_s"))
        .sort("_s", descending=True)
        .group_by("chrom")
        .agg(pl.col("GENE_SYMBOL").first().alias("top_gene")),
        on="chrom",
        how="left",
    )
    .with_columns(
        pl.col("chrom").replace_strict(_order, default=99).alias("_o")
    )
    .sort("_o")
    .drop("_o")
)

chrom_table = mo.ui.table(
    chrom_summary,
    selection="multi",
    page_size=25,
    label="**Chromosome** — select to filter the gene table",
)

In [ ]:
_csel = chrom_table.value
_chroms = _csel["chrom"].to_list() if len(_csel) else None
_shown = gene_index if _chroms is None else gene_index.filter(pl.col("chrom").is_in(_chroms))

gene_table = mo.ui.table(
    _shown,
    selection="single",
    page_size=15,
    label=(
        f"**Genes** — {_shown.height:,} shown"
        + ("" if _chroms is None else f", chromosome {', '.join(_chroms)}")
    ),
    format_mapping={
        "mgi_id": lambda v: mo.md(f"[{v}]({mgi_url(v)})" if v else "—"),
        "ncbi": lambda v: mo.md(f"[NCBI Gene]({v})"),
    },
)

mo.hstack([chrom_table, gene_table], widths=[1, 2], align="start")

<marimo-table data-initial-value='[]' data-label='"\u003cspan class=\"markdown prose dark:prose-invert contents\"\u003e\u003cspan class=\"paragraph\"\u003e\u003cstrong\u003eChromosome\u003c/strong\u003e — select to filter the gene table\u003c/span\u003e\u003c/span\u003e"' data-data='"[{\"_marimo_row_id\":0,\"chrom\":\"1\",\"genes\":999,\"strains\":3190,\"top_gene\":\"Enah\"},{\"_marimo_row_id\":1,\"chrom\":\"2\",\"genes\":1270,\"strains\":4374,\"top_gene\":\"a\"},{\"_marimo_row_id\":2,\"chrom\":\"3\",\"genes\":839,\"strains\":2542,\"top_gene\":\"Jade1\"},{\"_marimo_row_id\":3,\"chrom\":\"4\",\"genes\":1048,\"strains\":3453,\"top_gene\":\"Pum1\"},{\"_marimo_row_id\":4,\"chrom\":\"5\",\"genes\":983,\"strains\":3326,\"top_gene\":\"En2\"},{\"_marimo_row_id\":5,\"chrom\":\"6\",\"genes\":880,\"strains\":2851,\"top_gene\":\"Gt(ROSA)26Sor\"},{\"_marimo_row_id\":6,\"chrom\":\"7\",\"genes\":1243,\"strains\":3770,\"top_gene\":\"Ctbp2\"},{\"_marimo_row_id\":7,\"chrom\":\"8\",\"genes\":811,\"strains\":2392,\"top_gene\":\"Zfp423\"},{\"_marimo_row_id\":8,\"chrom\":\"9\",\"genes\":886,\"strains\":2806,\"top_gene\":\"DBH\"},{\"_marimo_row_id\":9,\"chrom\":\"10\",\"genes\":774,\"strains\":2481,\"top_gene\":\"Tet1\"},{\"_marimo_row_id\":10,\"chrom\":\"11\",\"genes\":1326,\"strains\":4569,\"top_gene\":\"Msi2\"},{\"_marimo_row_id\":11,\"chrom\":\"12\",\"genes\":577,\"strains\":1778,\"top_gene\":\"Cog5\"},{\"_marimo_row_id\":12,\"chrom\":\"13\",\"genes\":602,\"strains\":1956,\"top_gene\":\"Jarid2\"},{\"_marimo_row_id\":13,\"chrom\":\"14\",\"genes\":587,\"strains\":1828,\"top_gene\":\"Diaph3\"},{\"_marimo_row_id\":14,\"chrom\":\"15\",\"genes\":643,\"strains\":2267,\"top_gene\":\"Slc25a17\"},{\"_marimo_row_id\":15,\"chrom\":\"16\",\"genes\":522,\"strains\":1661,\"top_gene\":\"Lpp\"},{\"_marimo_row_id\":16,\"chrom\":\"17\",\"genes\":752,\"strains\":2407,\"top_gene\":\"HAP1\"},{\"_marimo_row_id\":17,\"chrom\":\"18\",\"genes\":393,\"strains\":1238,\"top_gene\":\"GRP\"},{\"_marimo_row_id\":18,\"chrom\":\"19\",\"genes\":572,\"strains\":1737,\"top_gene\":\"Snhg1\"},{\"_marimo_row_id\":19,\"chrom\":\"X\",\"genes\":629,\"strains\":2837,\"top_gene\":\"Hprt1\"},{\"_marimo_row_id\":20,\"chrom\":\"Y\",\"genes\":9,\"strains\":20,\"top_gene\":\"Ddx3y\"},{\"_marimo_row_id\":21,\"chrom\":\"MT\",\"genes\":1,\"strains\":13,\"top_gene\":\"mt-Rnr1\"},{\"_marimo_row_id\":22,\"chrom\":\"unmapped\",\"genes\":112,\"strains\":2223,\"top_gene\":\"EGFP\"}]"' data-total-rows='23' data-total-columns='4' data-max-columns='50' data-banner-text='""' data-pagination='false' data-page-size='25' data-field-types='[["chrom",["string","str"]],["genes",["integer","u32"]],["strains",["integer","u32"]],["top_gene",["string","str"]]]' data-selection='"multi"' data-show-filters='true' data-show-download='true' data-show-search='true' data-show-column-summaries='true' data-show-data-types='true' data-show-page-size-selector='true' data-show-column-explorer='true' data-show-chart-builder='true' data-row-headers='[]' data-hidden-columns='[]' data-has-stable-row-id='true' data-lazy='false' data-preload='false'> <marimo-table data-initial-value='[]' data-label='"\u003cspan class=\"markdown prose dark:prose-invert contents\"\u003e\u003cspan class=\"paragraph\"\u003e\u003cstrong\u003eGenes\u003c/strong\u003e — 16,426 shown\u003c/span\u003e\u003c/span\u003e"' data-data='"[{\"_marimo_row_id\":0,\"gene_symbol\":\"Hprt1\",\"gene_name\":\"hypoxanthine phosphoribosyltransferase 1\",\"chrom\":\"X\",\"strains\":1230,\"strains_all\":1233,\"alleles\":82,\"mutation_types\":\"TM\",\"mgi_id\":{\"_serialized_mime_bundle\":{\"mimetype\":\"text/markdown\",\"data\":\"[MGI:96217](https://www.informatics.jax.org/marker/MGI:96217)\"}},\"ncbi\":{\"_serialized_mime_bundle\":{\"mimetype\":\"text/markdown\",\"data\":\"[NCBI Gene](https://www.ncbi.nlm.nih.gov/gene/?term=Hprt1%5Bsym%5D+AND+%22Mus+musculus%22%5Borgn%5D)\"}}},{\"_marimo_row_id\":1,\"gene_symbol\":\"EGFP\",\"gene_name\":\"enhanced green fluorescent protein\",\

In [ ]:
_gsel = gene_table.value

if not len(_gsel):
    _output = mo.md("_Select a gene above to see its strains and cross-references._")
else:
    _g = _gsel["gene_symbol"][0]
    _mgi = _gsel["mgi_id"][0]
    _rows = catalog_genes.filter(
        pl.col("GENE_SYMBOL").eq(_g) & pl.col("mut").is_in(mutation_filter.value)
    )

    _links = [f"[NCBI Gene]({ncbi_url(_g, _mgi)})"]
    if _mgi:
        _links.insert(0, f"[{_mgi}]({mgi_url(_mgi)})")

    _by_mut = (
        _rows.group_by("mut")
        .agg(pl.col("STRAIN/STOCK_ID").n_unique().alias("strains"))
        .sort("strains", descending=True)
    )

    _strains = (
        _rows.unique(subset=["STRAIN/STOCK_ID"])
        .join(
            strain_alleles.group_by("STRAIN/STOCK_ID").agg(
                pl.col("ALLELE_SYMBOL").unique().sort().str.join(", ").alias("alleles")
            ),
            on="STRAIN/STOCK_ID",
            how="left",
        )
        .with_columns(
            pl.col("OTHER_NAMES").str.extract(r"(RRID:MMRRC_[\w.-]+)").alias("rrid"),
            pl.col("PUBMED_IDS").str.extract_all(r"\d{6,9}").alias("_pmids"),
        )
        .select([
            "STRAIN/STOCK_ID", "STRAIN/STOCK_DESIGNATION", "alleles",
            "mut", "STRAIN_TYPE", "STATE", "rrid", "_pmids", "SDS_URL",
        ])
        .rename({
            "STRAIN/STOCK_ID": "strain_id",
            "STRAIN/STOCK_DESIGNATION": "designation",
            "_pmids": "pubmed",
        })
        .sort("strain_id")
    )

    _output = mo.vstack([
        mo.md(
            f"### {_g} &nbsp; <small>{_gsel['gene_name'][0] or ''}</small>\n\n"
            f"Chromosome **{_gsel['chrom'][0]}** &nbsp;·&nbsp; "
            f"**{_gsel['strains'][0]:,}** strains under the current filter "
            f"({_gsel['strains_all'][0]:,} across all mutation types) &nbsp;·&nbsp; "
            f"**{_gsel['alleles'][0] or 0:,}** alleles\n\n"
            + " &nbsp;·&nbsp; ".join(_links)
        ),
        mo.md("**Strains by mutation type**"),
        mo.ui.table(_by_mut, selection=None, page_size=8),
        mo.md("**Strains carrying this gene**"),
        mo.ui.table(
            _strains,
            selection=None,
            page_size=10,
            format_mapping={
                "rrid": lambda v: mo.md(f"[{v}](https://scicrunch.org/resolver/{v})" if v else "—"),
                "SDS_URL": lambda v: mo.md(f"[data sheet]({v})" if v else "—"),
                "pubmed": lambda v: mo.md(
                    ", ".join(f"[{p}](https://pubmed.ncbi.nlm.nih.gov/{p}/)" for p in v)
                    if v is not None and len(v)
                    else "—"
                ),
            },
        ),
    ])

_output

_Select a gene above to see its strains and cross-references._

In [ ]:
_g = catalog_genes  # all gene rows, filter-independent
_ci = _g.filter(pl.col("mut") == "CI")
_non_ci = _g.filter(pl.col("mut") != "CI")
_n_genes = _g["GENE_SYMBOL"].n_unique()
_ci_only = _n_genes - _non_ci["GENE_SYMBOL"].n_unique()

_caps = pl.col("GENE_SYMBOL").str.contains(r"[a-z]").not_() & pl.col(
    "GENE_SYMBOL"
).str.contains(r"[A-Z]")
_sym = _g.select(["GENE_SYMBOL", "MGI_GENE_ACCESSION_ID"]).unique(subset=["GENE_SYMBOL"])

_top_ci = _ci.group_by("GENE_SYMBOL").agg(
    pl.col("STRAIN/STOCK_ID").n_unique().alias("s")
).sort("s", descending=True)["GENE_SYMBOL"].head(5).to_list()
_top_non_ci = _non_ci.group_by("GENE_SYMBOL").agg(
    pl.col("STRAIN/STOCK_ID").n_unique().alias("s")
).sort("s", descending=True)["GENE_SYMBOL"].head(5).to_list()

_by_chrom = _g.group_by("chrom").agg(
    pl.col("GENE_SYMBOL").n_unique().alias("genes"),
    pl.col("STRAIN/STOCK_ID").n_unique().alias("strains"),
)
_placed = _by_chrom.filter(pl.col("chrom") != "unmapped")
_most_genes = _placed.sort("genes", descending=True).row(0, named=True)
_most_strains = _placed.sort("strains", descending=True).row(0, named=True)

mo.md(f"""
## Things worth knowing about this data

**1. The top of the raw ranking is a gene-length artifact.** Ranking by strain
count with ENU included gives {", ".join(_top_ci)} — among the longest genes in the
mouse genome, hit most often by random mutagenesis simply for being the biggest
targets. Worse, **{_ci_only:,} of {_n_genes:,} genes
({_ci_only / _n_genes:.0%}) appear *only* in chemically-induced strains** — they
have never been deliberately manipulated in this catalog at all.

**2. The most-manipulated "genes" are reagents, not disease genes.** Excluding ENU,
the ranking is {", ".join(_top_non_ci)} — the toolkit of mouse genetics.
`Hprt1` is the ES-cell HAT-selection locus (and sits on the X); `EGFP`, `cre`,
`lacZ` and `tTA` are cassettes with **no MGI id and no chromosome**, so they land
in the `unmapped` bucket. `Gt(ROSA)26Sor` is the canonical safe-harbour locus.
Genuine biology starts several rows down.

**3. Human transgenes are hiding in the gene column.**
{_sym.filter(_caps).height:,} symbols are ALL-CAPS
({_sym.filter(_caps & pl.col("MGI_GENE_ACCESSION_ID").is_null()).height:,} of them
with no MGI id) — non-mouse genes carried as transgenes. The giveaway is that some
sit on "chromosome" 20, 21 and 22, which mice do not have: `SOD1` and `APP` on
chr21 are *human* coordinates, from ALS and Alzheimer models. The NCBI links above
drop the mouse organism filter for these so they still resolve.

**4. `CHROMOSOME` is free text, not a category.**
{catalog["CHROMOSOME"].n_unique():,} distinct values for what should be 22,
including `unknown`, `UN`, `unk`, `N/A`, `Chr 1`, `Chr11:4938754-4948064 bp`,
`8q21.13` (a *human* cytoband), `919`, and one entry where chromosome 14 was
typed with a stray backtick. Normalisation recovers most of it and leaves
{_by_chrom.filter(pl.col("chrom") == "unmapped")["strains"].sum():,} strains
unmapped — but that bucket is not only dirt, it is also where the reagent
cassettes legitimately live.

**5. Coverage is uneven across the genome.** Chromosome {_most_genes["chrom"]}
carries the most distinct genes ({_most_genes["genes"]:,}) while chromosome
{_most_strains["chrom"]} carries the most strains ({_most_strains["strains"]:,}) —
gene density and research attention are not the same thing.
""")

## Things worth knowing about this data

**1. The top of the raw ranking is a gene-length artifact.** Ranking by strain
count with ENU included gives Ttn, Obscn, Neb, Rsf1, Dst — among the longest genes in the
mouse genome, hit most often by random mutagenesis simply for being the biggest
targets. Worse, **8,167 of 24,593 genes
(33%) appear *only* in chemically-induced strains** — they
have never been deliberately manipulated in this catalog at all.

**2. The most-manipulated "genes" are reagents, not disease genes.** Excluding ENU,
the ranking is Hprt1, EGFP, cre, lacZ, Ctbp2 — the toolkit of mouse genetics.
`Hprt1` is the ES-cell HAT-selection locus (and sits on the X); `EGFP`, `cre`,
`lacZ` and `tTA` are cassettes with **no MGI id and no chromosome**, so they land
in the `unmapped` bucket. `Gt(ROSA)26Sor` is the canonical safe-harbour locus.
Genuine biology starts several rows down.

**3. Human transgenes are hiding in the gene column.**
723 symbols are ALL-CAPS
(592 of them
with no MGI id) — non-mouse genes carried as transgenes. The giveaway is that some
sit on "chromosome" 20, 21 and 22, which mice do not have: `SOD1` and `APP` on
chr21 are *human* coordinates, from ALS and Alzheimer models. The NCBI links above
drop the mouse organism filter for these so they still resolve.

**4. `CHROMOSOME` is free text, not a category.**
50 distinct values for what should be 22,
including `unknown`, `UN`, `unk`, `N/A`, `Chr 1`, `Chr11:4938754-4948064 bp`,
`8q21.13` (a *human* cytoband), `919`, and one entry where chromosome 14 was
typed with a stray backtick. Normalisation recovers most of it and leaves
2,226 strains
unmapped — but that bucket is not only dirt, it is also where the reagent
cassettes legitimately live.

**5. Coverage is uneven across the genome.** Chromosome 7
carries the most distinct genes (2,202) while chromosome
11 carries the most strains (12,113) —
gene density and research attention are not the same thing.

# Phenotype exploration

`MPT_IDS` holds Mammalian Phenotype annotations as pipe-separated
`label [MP:id]` pairs:

> `decreased bone mineral density [MP:0000063]| abnormal vertebrae morphology [MP:0000137]| …`

This section cross-links them against `downloaded/mp.owl.gz`, the Mammalian Phenotype
Ontology, to group phenotypes under their parent categories and see which areas
of mouse biology the collection actually covers.

**Only the ids in that column are usable.** MMRRC's export re-splits the joined
label string on `", "`, so any phenotype name containing a comma is torn in two
and the tail of each affected list is lost. The ids are untouched, so the cells
below read ids only and take every label from the ontology. The highlights at the
end of the section show the evidence
([issue #1](https://github.com/gaurav/mmrrc/issues/1)).

In [ ]:
MP_OWL = Path("../downloaded/mp.owl.gz")
MP_ROOT = "MP:0000001"

_OBO = "http://purl.obolibrary.org/obo/"
_RDF = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}"
_RDFS = "{http://www.w3.org/2000/01/rdf-schema#}"
_OWL = "{http://www.w3.org/2002/07/owl#}"

mp_labels = {}
mp_obsolete = set()
mp_parents = defaultdict(list)

# One streaming pass over 101 MB of RDF/XML (5 MB gzipped), ~1.5s -- no ontology
# library needed. Only clear owl:Class elements: clearing every element wipes
# child text before the parent's end event can read it.
with gzip.open(MP_OWL) as _fh:
    for _ev, _el in ET.iterparse(_fh, events=("end",)):
        if _el.tag != _OWL + "Class":
            continue
        _about = _el.get(_RDF + "about", "")
        if _about.startswith(_OBO + "MP_"):
            _cid = "MP:" + _about.rsplit("MP_", 1)[1]
            _label = _el.findtext(_RDFS + "label")
            if _label:
                mp_labels[_cid] = _label
            if _el.findtext(_OWL + "deprecated") == "true":
                mp_obsolete.add(_cid)
            for _sub in _el.findall(_RDFS + "subClassOf"):
                # Named parents only; anonymous owl:Restriction children carry no
                # rdf:resource and are skipped.
                _r = _sub.get(_RDF + "resource")
                if _r and _r.startswith(_OBO + "MP_"):
                    mp_parents[_cid].append("MP:" + _r.rsplit("MP_", 1)[1])
        _el.clear()

# The 28 children of "mammalian phenotype" -- the body-system grouping.
mp_categories = {
    _c: mp_labels[_c]
    for _c in sorted(
        (_k for _k, _ps in mp_parents.items() if MP_ROOT in _ps),
        key=lambda _k: mp_labels[_k],
    )
}


def mp_ancestors(mp_id):
    """Every ancestor of a term. MP is a DAG, so a term can have several parents."""
    _seen, _out, _q = {mp_id}, set(), deque([mp_id])
    while _q:
        for _p in mp_parents.get(_q.popleft(), ()):
            if _p not in _seen:
                _seen.add(_p)
                _out.add(_p)
                _q.append(_p)
    return _out


_cat_ids = set(mp_categories)
mp_rollup = pl.DataFrame(
    [
        {"mp_id": _c, "category_id": _k}
        for _c in mp_labels
        for _k in ({_c} | mp_ancestors(_c)) & _cat_ids
    ],
    schema={"mp_id": pl.String, "category_id": pl.String},
)

mo.md(
    f"`mp.owl`: **{len(mp_labels):,} MP terms** ({len(mp_obsolete)} obsolete), "
    f"**{len(mp_categories)} top-level categories**, "
    f"{mp_rollup.height:,} term→category edges "
    f"({mp_rollup.height / mp_rollup['mp_id'].n_unique():.2f} categories per term — "
    f"MP is a DAG, so these overlap)."
)

`mp.owl`: **15,288 MP terms** (457 obsolete), **28 top-level categories**, 22,082 term→category edges (1.49 categories per term — MP is a DAG, so these overlap).

In [ ]:
# Take the MP ids and nothing else. The label half of MPT_IDS is unusable:
# MMRRC joins the phenotype names into one comma-separated string, re-splits it on
# ", " and truncates to the number of ids -- so any label containing a comma is
# torn in two and the tail of the list is dropped. The ids are untouched: complete,
# correctly ordered, and every one a valid MP term. Labels come from the ontology.
# See https://github.com/gaurav/mmrrc/issues/1.
strain_phenotypes = (
    catalog.filter(pl.col("MPT_IDS").is_not_null())
    .unique(subset=["STRAIN/STOCK_ID"])
    .select(
        pl.col("STRAIN/STOCK_ID").alias("strain_id"),
        pl.col("MPT_IDS").str.extract_all(r"MP:\d+").alias("mp_id"),
    )
    .explode("mp_id", empty_as_null=False)
    .unique()
)

# Diagnostics for the highlights below: how badly the label half is mangled, and
# confirmation that the ids are the intact half.
_dmg = dict(strains=0, ids=0, truncated_strains=0, lost_slots=0, pos=0, pos_match=0)
for _row in (
    catalog.filter(pl.col("MPT_IDS").is_not_null())
    .unique(subset=["STRAIN/STOCK_ID"])
    .select(["MPT_IDS"])
    .iter_rows(named=True)
):
    _entries = [_e.strip() for _e in _row["MPT_IDS"].split("|")]
    _ids = re.findall(r"MP:\d+", _row["MPT_IDS"])
    _texts = [re.sub(r"\s*\[MP:\d+\]\s*$", "", _e) for _e in _entries]
    _dmg["strains"] += 1
    _dmg["ids"] += len(_ids)
    if any(_i not in mp_labels for _i in _ids):
        continue
    # Reconstruct what MMRRC's exporter did: join the real labels, re-split on
    # ", ", keep only as many pieces as there are ids.
    _expanded = ", ".join(mp_labels[_i] for _i in _ids).split(", ")
    if len(_expanded) > len(_ids):
        _dmg["truncated_strains"] += 1
        _dmg["lost_slots"] += len(_expanded) - len(_ids)
    for _a, _b in zip(_texts, _expanded):
        _dmg["pos"] += 1
        _dmg["pos_match"] += _a.lower() == _b.lower()

mp_label_damage = _dmg

mo.md(
    f"`strain_phenotypes`: **{strain_phenotypes.height:,} strain→phenotype edges** "
    f"over {strain_phenotypes['strain_id'].n_unique():,} strains and "
    f"{strain_phenotypes['mp_id'].n_unique():,} distinct MP terms, taken from the ids "
    f"alone. Reconstructing MMRRC's mangled label column from those ids reproduces "
    f"**{_dmg['pos_match'] / _dmg['pos']:.1%}** of its {_dmg['pos']:,} label slots — "
    f"the ids are the intact half."
)

`strain_phenotypes`: **47,223 strain→phenotype edges** over 4,604 strains and 5,657 distinct MP terms, taken from the ids alone. Reconstructing MMRRC's mangled label column from those ids reproduces **91.3%** of its 48,069 label slots — the ids are the intact half.

In [ ]:
def mp_url(mp_id):
    """Link to the MGI Mammalian Phenotype browser for a term."""
    return f"https://www.informatics.jax.org/vocab/mp_ontology/{mp_id}"


_cat_names = pl.DataFrame(
    {"category_id": list(mp_categories), "category": list(mp_categories.values())}
)

_term_cats = (
    mp_rollup.join(_cat_names, on="category_id", how="inner")
    .group_by("mp_id")
    .agg(pl.col("category").unique().sort().str.join(", ").alias("categories"))
)

# Genes reach a phenotype through the strain, exactly as alleles do (see the
# grain notes) -- catalog_genes unfiltered, so this does not inherit the gene
# section's mutation-type filter.
_strain_genes = (
    catalog_genes.select(["STRAIN/STOCK_ID", "GENE_SYMBOL"])
    .unique()
    .rename({"STRAIN/STOCK_ID": "strain_id"})
)

phenotype_index = (
    strain_phenotypes.group_by("mp_id")
    .agg(pl.col("strain_id").n_unique().alias("strains"))
    .join(
        strain_phenotypes.join(_strain_genes, on="strain_id", how="inner")
        .group_by("mp_id")
        .agg(pl.col("GENE_SYMBOL").n_unique().alias("genes")),
        on="mp_id",
        how="left",
    )
    .join(_term_cats, on="mp_id", how="left")
    .with_columns(
        pl.col("mp_id").replace_strict(mp_labels, default=None).alias("label"),
        pl.col("mp_id").is_in(list(mp_obsolete)).alias("obsolete"),
        pl.col("genes").fill_null(0),
    )
    .select(["label", "mp_id", "strains", "genes", "categories", "obsolete"])
    .sort("strains", descending=True)
)

mo.md(
    f"`phenotype_index`: **{phenotype_index.height:,} distinct phenotypes** used by "
    f"{strain_phenotypes['strain_id'].n_unique():,} of "
    f"{catalog['STRAIN/STOCK_ID'].n_unique():,} strains "
    f"({strain_phenotypes['strain_id'].n_unique() / catalog['STRAIN/STOCK_ID'].n_unique():.1%})."
)

`phenotype_index`: **5,657 distinct phenotypes** used by 4,604 of 69,388 strains (6.6%).

In [ ]:
_cat_names2 = pl.DataFrame(
    {"category_id": list(mp_categories), "category": list(mp_categories.values())}
)

category_summary = (
    strain_phenotypes.join(mp_rollup, on="mp_id", how="inner")
    .group_by("category_id")
    .agg(
        pl.col("mp_id").n_unique().alias("phenotypes"),
        pl.col("strain_id").n_unique().alias("strains"),
    )
    .join(_cat_names2, on="category_id", how="inner")
    .select(["category", "strains", "phenotypes", "category_id"])
    .sort("strains", descending=True)
)

category_table = mo.ui.table(
    category_summary,
    selection="multi",
    page_size=30,
    label=(
        "**MP category** — select to filter. A term can sit under several "
        "categories, so these columns overlap and do not sum to the total."
    ),
)

In [ ]:
_csel = category_table.value
_cats = _csel["category_id"].to_list() if len(_csel) else None

if _cats is None:
    _shown = phenotype_index
else:
    _ids = mp_rollup.filter(pl.col("category_id").is_in(_cats))["mp_id"].unique().to_list()
    _shown = phenotype_index.filter(pl.col("mp_id").is_in(_ids))

phenotype_table = mo.ui.table(
    _shown,
    selection="single",
    page_size=15,
    label=(
        f"**Phenotypes** — {_shown.height:,} shown"
        + ("" if _cats is None else ", filtered by category")
    ),
    format_mapping={
        "mp_id": lambda v: mo.md(f"[{v}]({mp_url(v)})" if v else "—"),
    },
)

mo.hstack([category_table, phenotype_table], widths=[1, 2], align="start")

<marimo-table data-initial-value='[]' data-label='"\u003cspan class=\"markdown prose dark:prose-invert contents\"\u003e\u003cspan class=\"paragraph\"\u003e\u003cstrong\u003eMP category\u003c/strong\u003e — select to filter. A term can sit under several categories, so these columns overlap and do not sum to the total.\u003c/span\u003e\u003c/span\u003e"' data-data='"[{\"_marimo_row_id\":0,\"category\":\"homeostasis/metabolism phenotype\",\"strains\":1730,\"phenotypes\":560,\"category_id\":\"MP:0005376\"},{\"_marimo_row_id\":1,\"category\":\"mortality/aging\",\"strains\":1566,\"phenotypes\":71,\"category_id\":\"MP:0010768\"},{\"_marimo_row_id\":2,\"category\":\"growth/size/body region phenotype\",\"strains\":1465,\"phenotypes\":385,\"category_id\":\"MP:0005378\"},{\"_marimo_row_id\":3,\"category\":\"hematopoietic system phenotype\",\"strains\":1241,\"phenotypes\":539,\"category_id\":\"MP:0005397\"},{\"_marimo_row_id\":4,\"category\":\"immune system phenotype\",\"strains\":1167,\"phenotypes\":736,\"category_id\":\"MP:0005387\"},{\"_marimo_row_id\":5,\"category\":\"behavior/neurological phenotype\",\"strains\":1140,\"phenotypes\":253,\"category_id\":\"MP:0005386\"},{\"_marimo_row_id\":6,\"category\":\"cellular phenotype\",\"strains\":918,\"phenotypes\":404,\"category_id\":\"MP:0005384\"},{\"_marimo_row_id\":7,\"category\":\"nervous system phenotype\",\"strains\":868,\"phenotypes\":721,\"category_id\":\"MP:0003631\"},{\"_marimo_row_id\":8,\"category\":\"skeleton phenotype\",\"strains\":837,\"phenotypes\":550,\"category_id\":\"MP:0005390\"},{\"_marimo_row_id\":9,\"category\":\"cardiovascular system phenotype\",\"strains\":729,\"phenotypes\":552,\"category_id\":\"MP:0005385\"},{\"_marimo_row_id\":10,\"category\":\"integument phenotype\",\"strains\":638,\"phenotypes\":278,\"category_id\":\"MP:0010771\"},{\"_marimo_row_id\":11,\"category\":\"vision/eye phenotype\",\"strains\":609,\"phenotypes\":280,\"category_id\":\"MP:0005391\"},{\"_marimo_row_id\":12,\"category\":\"endocrine/exocrine gland phenotype\",\"strains\":581,\"phenotypes\":437,\"category_id\":\"MP:0005379\"},{\"_marimo_row_id\":13,\"category\":\"reproductive system phenotype\",\"strains\":529,\"phenotypes\":316,\"category_id\":\"MP:0005389\"},{\"_marimo_row_id\":14,\"category\":\"normal phenotype\",\"strains\":444,\"phenotypes\":2,\"category_id\":\"MP:0002873\"},{\"_marimo_row_id\":15,\"category\":\"craniofacial phenotype\",\"strains\":413,\"phenotypes\":396,\"category_id\":\"MP:0005382\"},{\"_marimo_row_id\":16,\"category\":\"limbs/digits/tail phenotype\",\"strains\":399,\"phenotypes\":126,\"category_id\":\"MP:0005371\"},{\"_marimo_row_id\":17,\"category\":\"embryo phenotype\",\"strains\":364,\"phenotypes\":261,\"category_id\":\"MP:0005380\"},{\"_marimo_row_id\":18,\"category\":\"adipose tissue phenotype\",\"strains\":361,\"phenotypes\":72,\"category_id\":\"MP:0005375\"},{\"_marimo_row_id\":19,\"category\":\"liver/biliary system phenotype\",\"strains\":352,\"phenotypes\":78,\"category_id\":\"MP:0005370\"},{\"_marimo_row_id\":20,\"category\":\"pigmentation phenotype\",\"strains\":333,\"phenotypes\":64,\"category_id\":\"MP:0001186\"},{\"_marimo_row_id\":21,\"category\":\"hearing/vestibular/ear phenotype\",\"strains\":314,\"phenotypes\":210,\"category_id\":\"MP:0005377\"},{\"_marimo_row_id\":22,\"category\":\"digestive/alimentary phenotype\",\"strains\":312,\"phenotypes\":269,\"category_id\":\"MP:0005381\"},{\"_marimo_row_id\":23,\"category\":\"muscle phenotype\",\"strains\":310,\"phenotypes\":199,\"category_id\":\"MP:0005369\"},{\"_marimo_row_id\":24,\"category\":\"respiratory system phenotype\",\"strains\":293,\"phenotypes\":198,\"category_id\":\"MP:0005388\"},{\"_marimo_row_id\":25,\"category\":\"renal/urinary system phenotype\",\"strains\":285,\"phenotypes\":214,\"category_id\":\"MP:0005367\"},{\"_marimo_row_id\":26,\"category\":\"neoplasm\",\"strains\":226,\"phenotypes\":148,\"category_id\":\"MP:0002006\"},{\"_marimo_row_id\":27,\"category\":\"taste/olfaction phenotype\",\

In [ ]:
_psel = phenotype_table.value

if not len(_psel):
    _output = mo.md("_Select a phenotype above to see its genes and strains._")
else:
    _mp = _psel["mp_id"][0]
    _strain_ids = strain_phenotypes.filter(pl.col("mp_id") == _mp)["strain_id"].to_list()

    _genes = (
        catalog_genes.filter(pl.col("STRAIN/STOCK_ID").is_in(_strain_ids))
        .group_by("GENE_SYMBOL")
        .agg(
            pl.col("STRAIN/STOCK_ID").n_unique().alias("strains"),
            pl.col("MGI_GENE_ACCESSION_ID").drop_nulls().first().alias("mgi_id"),
            pl.col("chrom").first().alias("chrom"),
        )
        .rename({"GENE_SYMBOL": "gene_symbol"})
        .sort("strains", descending=True)
    )

    _strains = (
        catalog.filter(pl.col("STRAIN/STOCK_ID").is_in(_strain_ids))
        .unique(subset=["STRAIN/STOCK_ID"])
        .with_columns(
            pl.col("OTHER_NAMES").str.extract(r"(RRID:MMRRC_[\w.-]+)").alias("rrid")
        )
        .select([
            "STRAIN/STOCK_ID", "STRAIN/STOCK_DESIGNATION",
            "MUTATION_TYPE", "STRAIN_TYPE", "STATE", "rrid", "SDS_URL",
        ])
        .rename({
            "STRAIN/STOCK_ID": "strain_id",
            "STRAIN/STOCK_DESIGNATION": "designation",
            "MUTATION_TYPE": "mut",
        })
        .sort("strain_id")
    )

    _output = mo.vstack([
        mo.md(
            f"### {_psel['label'][0]}\n\n"
            f"[{_mp}]({mp_url(_mp)}) &nbsp;·&nbsp; "
            f"**{_psel['strains'][0]:,}** strains &nbsp;·&nbsp; "
            f"**{_psel['genes'][0]:,}** genes &nbsp;·&nbsp; "
            f"_{_psel['categories'][0] or 'no category'}_"
        ),
        mo.md("**Genes most associated with this phenotype**"),
        mo.ui.table(
            _genes,
            selection=None,
            page_size=8,
            format_mapping={
                "mgi_id": lambda v: mo.md(f"[{v}]({mgi_url(v)})" if v else "—"),
                "gene_symbol": lambda v: mo.md(f"[{v}]({ncbi_url(v)})"),
            },
        ),
        mo.md("**Strains showing it**"),
        mo.ui.table(
            _strains,
            selection=None,
            page_size=10,
            format_mapping={
                "rrid": lambda v: mo.md(
                    f"[{v}](https://scicrunch.org/resolver/{v})" if v else "—"
                ),
                "SDS_URL": lambda v: mo.md(f"[data sheet]({v})" if v else "—"),
            },
        ),
    ])

_output

_Select a phenotype above to see its genes and strains._

In [ ]:
_d = mp_label_damage
_all_strains = catalog["STRAIN/STOCK_ID"].n_unique()
_ann_ids = strain_phenotypes["strain_id"].unique().to_list()
_ann = len(_ann_ids)

_mut = (
    catalog.filter(pl.col("STRAIN/STOCK_ID").is_in(_ann_ids))
    .group_by("STRAIN/STOCK_ID")
    .agg(pl.col("MUTATION_TYPE").drop_nulls().unique().sort().str.join("+").alias("mut"))
    .group_by("mut")
    .agg(pl.len().alias("n"))
)
_tm = _mut.filter(pl.col("mut") == "TM")["n"].sum()
_ci = _mut.filter(pl.col("mut") == "CI")["n"].sum()
_top2 = phenotype_index.head(2)

mo.md(f"""
## Things worth knowing about the phenotype data

**1. The labels in `MPT_IDS` are corrupt; the ids are fine. Use the ids.**
MMRRC's exporter joins the phenotype names into one comma-separated string,
re-splits that string on `", "`, and zips the pieces against the id list,
truncating to its length. Because MP labels legitimately contain commas
(`decreased CD4-positive, alpha-beta T cell number`), every such label is torn in
two, everything after it slides by one, and the tail of the label list falls off
the end.

Reconstructing the column from the ids alone — join the ontology's labels, split
on `", "`, truncate — reproduces **{_d["pos_match"]:,} of {_d["pos"]:,}
({_d["pos_match"] / _d["pos"]:.1%})** of the label slots in the file, which is
what identifies the mechanism. The residual is ordinary label drift (point 2).

{_d["truncated_strains"]:,} of {_d["strains"]:,} annotated strains are affected,
losing {_d["lost_slots"]:,} label slots. All {_d["ids"]:,} ids are valid MP terms.

`MMRRC:011644-UNC` is the shape of it — four ids, but only enough label text for
the first four fragments of three of them:

| Entry in the file | The id is right | The label beside it is not |
|---|---|---|
| `abnormal trophoblast giant cell morphology [MP:0005033]` | abnormal trophoblast giant cell morphology | ✅ |
| `embryonic lethality between implantation and somite formation [MP:0011096]` | …, **complete penetrance** | truncated at the comma |
| `complete penetrance [MP:0011100]` | preweaning lethality, complete penetrance | the other half of the line above |
| `preweaning lethality [MP:0012113]` | **decreased inner cell mass proliferation** | label ran out; text is a leftover |

The strain really does have `MP:0012113 decreased inner cell mass proliferation` —
its name never appears in the file because the label expansion was cut short.

**2. The labels are also a stale snapshot.** Independent of the mangling, MMRRC's
text lags the ontology — `hypoactivity` is now `decreased locomotor activity`,
`retinal degeneration` is now `retina degeneration`, `thyroid inflammation` is now
`thyroid gland inflammation`, `aorta dilation` is now `dilated aorta`. Another
reason to take ids and render labels from `mp.owl`, as every table above does.

**3. Phenotype coverage is thin and skewed.** Only **{_ann:,} of
{_all_strains:,} strains ({_ann / _all_strains:.1%})** carry any phenotype, and
they lean toward deliberately characterised lines — {_tm:,} purely
targeted-mutation strains against {_ci:,} chemically-induced, in a catalog whose
largest single group is gene traps. Only
{strain_phenotypes["mp_id"].n_unique():,} of {len(mp_labels):,} MP terms
({strain_phenotypes["mp_id"].n_unique() / len(mp_labels):.0%}) are used at all.
Absence of a phenotype here means absence of *characterisation*, never absence of
an effect.

**4. The second most common "phenotype" is the absence of one.**
`{_top2["label"][1]}` ({_top2["mp_id"][1]}) sits on {_top2["strains"][1]:,}
strains, just behind `{_top2["label"][0]}` at {_top2["strains"][0]:,}. It is a
negative result, not a phenotype. It is left in the table — the ontology files it
under *normal phenotype*, so the category column flags it — but any ranking that
treats it as a finding is wrong.

**5. Categories overlap by design.** MP is a DAG, not a tree:
{mp_rollup.height:,} term→category edges across
{mp_rollup["mp_id"].n_unique():,} terms, a mean of
{mp_rollup.height / mp_rollup["mp_id"].n_unique():.2f} categories per term and up
to 5. One strain with one phenotype can count toward several categories, so the
category table's columns never sum to the totals.
""")

## Things worth knowing about the phenotype data

**1. The labels in `MPT_IDS` are corrupt; the ids are fine. Use the ids.**
MMRRC's exporter joins the phenotype names into one comma-separated string,
re-splits that string on `", "`, and zips the pieces against the id list,
truncating to its length. Because MP labels legitimately contain commas
(`decreased CD4-positive, alpha-beta T cell number`), every such label is torn in
two, everything after it slides by one, and the tail of the label list falls off
the end.

Reconstructing the column from the ids alone — join the ontology's labels, split
on `", "`, truncate — reproduces **43,900 of 48,069
(91.3%)** of the label slots in the file, which is
what identifies the mechanism. The residual is ordinary label drift (point 2).

1,450 of 4,604 annotated strains are affected,
losing 2,148 label slots. All 48,069 ids are valid MP terms.

`MMRRC:011644-UNC` is the shape of it — four ids, but only enough label text for
the first four fragments of three of them:

| Entry in the file | The id is right | The label beside it is not |
|---|---|---|
| `abnormal trophoblast giant cell morphology [MP:0005033]` | abnormal trophoblast giant cell morphology | ✅ |
| `embryonic lethality between implantation and somite formation [MP:0011096]` | …, **complete penetrance** | truncated at the comma |
| `complete penetrance [MP:0011100]` | preweaning lethality, complete penetrance | the other half of the line above |
| `preweaning lethality [MP:0012113]` | **decreased inner cell mass proliferation** | label ran out; text is a leftover |

The strain really does have `MP:0012113 decreased inner cell mass proliferation` —
its name never appears in the file because the label expansion was cut short.

**2. The labels are also a stale snapshot.** Independent of the mangling, MMRRC's
text lags the ontology — `hypoactivity` is now `decreased locomotor activity`,
`retinal degeneration` is now `retina degeneration`, `thyroid inflammation` is now
`thyroid gland inflammation`, `aorta dilation` is now `dilated aorta`. Another
reason to take ids and render labels from `mp.owl`, as every table above does.

**3. Phenotype coverage is thin and skewed.** Only **4,604 of
69,388 strains (6.6%)** carry any phenotype, and
they lean toward deliberately characterised lines — 2,666 purely
targeted-mutation strains against 374 chemically-induced, in a catalog whose
largest single group is gene traps. Only
5,657 of 15,288 MP terms
(37%) are used at all.
Absence of a phenotype here means absence of *characterisation*, never absence of
an effect.

**4. The second most common "phenotype" is the absence of one.**
`no abnormal phenotype detected` (MP:0002169) sits on 442
strains, just behind `decreased body weight` at 448. It is a
negative result, not a phenotype. It is left in the table — the ontology files it
under *normal phenotype*, so the category column flags it — but any ranking that
treats it as a finding is wrong.

**5. Categories overlap by design.** MP is a DAG, not a tree:
22,082 term→category edges across
14,830 terms, a mean of
1.49 categories per term and up
to 5. One strain with one phenotype can count toward several categories, so the
category table's columns never sum to the totals.